# Environment check

Run this **on the computer you will use for the test**, before the test day. Run every cell. If all three tests print results without errors, the machine is ready for the test.

If a package is missing, open **Anaconda Prompt** and run:

```
pip install pyomo simpy "ortools>=9.15,<9.16" "highspy>=1.11,<1.13"
```

(OR-Tools is not available from conda on Windows, and newer HiGHS versions clash with it.)

then restart the kernel and run this notebook again.

On the course website (browser) the Pyomo and SimPy checks work, but the OR-Tools check reports that OR-Tools is missing: it has no browser version, so use a computer for the test. The same checks run as `python environment_check.py`.

In [ ]:
# Run this cell first.
# On the course website (JupyterLite) it installs this notebook's packages into the browser tab
# (about a minute the first time) and adapts them to the browser.
# On your own computer it does nothing: install everything once with
#     pip install -r requirements.txt      (the file is at the top of the Assignments folder)
# In Google Colab, run  !pip install pyomo highspy simpy  instead.
import sys
if sys.platform == "emscripten":  # only in the browser
    import micropip
    await micropip.install(["pyomo==6.10.1", "highspy", "simpy==4.1.2"])
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True
print("Ready.")

## 1. Pyomo + HiGHS

In [ ]:
from pyomo.environ import *

# Model
model = ConcreteModel()

# Variable
model.x = Var(domain=NonNegativeReals)

# Objective
model.obj = Objective(expr=2 * model.x, sense=maximize)

# Constraint
model.con = Constraint(expr=model.x <= 5)

# Solve
solver = SolverFactory("appsi_highs")
solver.solve(model, tee=False)

# Display the model
model.pprint()

# Display the results
model.display()

## 2. Google OR-Tools (CP-SAT)

In [ ]:
import importlib

try:
    cp_model = importlib.import_module("ortools.sat.python.cp_model")
except ImportError:
    cp_model = None
    print("OR-Tools is NOT installed. On your computer run:")
    print('    pip install "ortools>=9.15,<9.16" "highspy>=1.11,<1.13"')
    print("(OR-Tools has no browser version, so this message is expected on the course website.)")

if cp_model is not None:
    """Minimal CP-SAT example to showcase calling the solver."""
    # Creates the model.
    model = cp_model.CpModel()

    # Creates the variables.
    var_upper_bound = max(50, 45, 37)
    x = model.new_int_var(0, var_upper_bound, "x")
    y = model.new_int_var(0, var_upper_bound, "y")
    z = model.new_int_var(0, var_upper_bound, "z")

    # Creates the constraints.
    model.add(2 * x + 7 * y + 3 * z <= 50)
    model.add(3 * x - 5 * y + 7 * z <= 45)
    model.add(5 * x + 2 * y - 6 * z <= 37)

    model.maximize(2 * x + 2 * y + 3 * z)

    # Creates a solver and solves the model.
    solver = cp_model.CpSolver()
    status = solver.solve(model)

    if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
        print(f"Maximum of objective function: {solver.objective_value}\n")
        print(f"x = {solver.value(x)}")
        print(f"y = {solver.value(y)}")
        print(f"z = {solver.value(z)}")
    else:
        print("No solution found.")

## 3. SimPy

In [ ]:
import simpy

def simple_process(env):
    print("Start at", env.now)
    yield env.timeout(5)
    print("End at", env.now)

env = simpy.Environment()
env.process(simple_process(env))
env.run()